In [15]:
import os
import time
from pysat.formula import CNF

from pysat.solvers import Kissat404
from partitionsolver.utils import file_reader


In [16]:
def get_combined_variable(lit1, lit2, var_amount):
    var1 = abs(lit1)
    var2 = abs(lit2)
    assert var1 != var2, "Variables must be different"
    smaller_var = min(var1, var2)
    larger_var = max(var1, var2)
    smaller_lit = lit1 if var1 == smaller_var else lit2
    larger_lit = lit1 if var1 == larger_var else lit2

    sign_offset = 0
    if (smaller_lit > 0 and larger_lit > 0):
        sign_offset = 0
    elif (smaller_lit > 0 and larger_lit < 0):
        sign_offset = 1
    elif (smaller_lit < 0 and larger_lit > 0):
        sign_offset = 2
    else:
        sign_offset = 3
    return sign_offset * var_amount * var_amount + smaller_var * var_amount + larger_var

def create_2CNF_clauses(clause, var_amount):
    assert len(clause) == 3, "Clause must have exactly 3 elements - assuming 3CNF input"
    combined_clauses = []
    alpha_1 = get_combined_variable(clause[0], clause[1], var_amount)
    alpha_2 = get_combined_variable(clause[0], clause[2], var_amount)
    alpha_3 = get_combined_variable(clause[1], clause[2], var_amount)

    combined_clauses.append([alpha_1, alpha_2])
    combined_clauses.append([alpha_1, alpha_3])
    combined_clauses.append([alpha_2, alpha_3])
    return combined_clauses

def create_2CNF_constraints(var_amount):
    constraints = []
    for variable in range(1, var_amount + 1):
        for other_variable in range(variable + 1, var_amount + 1):
            alpha_1 = get_combined_variable(variable, other_variable, var_amount)
            alpha_2 = get_combined_variable(-variable, other_variable, var_amount)
            alpha_3 = get_combined_variable(variable, -other_variable, var_amount)
            alpha_4 = get_combined_variable(-variable, -other_variable, var_amount)

            # See count_sat_assignments: Exactly one must be false
            # -(-alpha_1 and -alpha_2) = alpha_1 or alpha_2
            constraints.append([alpha_1, alpha_2])
            constraints.append([alpha_1, alpha_3])
            constraints.append([alpha_1, alpha_4])
            constraints.append([alpha_2, alpha_3])
            constraints.append([alpha_2, alpha_4])
            constraints.append([alpha_3, alpha_4])
    return constraints


def convert_3CNF_to_2CNF(clauses, var_amount):
    new_clauses = []
    for clause in clauses:
        new_clauses.extend(create_2CNF_clauses(clause, var_amount))
    
    new_clauses.extend(create_2CNF_constraints(var_amount))
    return new_clauses

In [ ]:
def count_sat_assignments(alpha_1, alpha_2, alpha_3, alpha_4):
    count = 0
    for a in [True, False]:
        for b in [True, False]:
            sat = True
            # alpha_1 = (a or b)
            if alpha_1:
                sat &= (a or b)
            else:
                sat &= not (a or b)

            # alpha_2 = (not a or b)
            if alpha_2:
                sat &= (not a or b)
            else:
                sat &= not (not a or b)

            # alpha_3 = (a or not b)
            if alpha_3:
                sat &= (a or not b)
            else:
                sat &= not (a or not b)

            # alpha_4 = (not a or not b)
            if alpha_4:
                sat &= (not a or not b)
            else:
                sat &= not (not a or not b)
            if sat:
                count += 1
    return count

def test_constraints(variabe, other_variable, var_amount):
    truth_table = []
    for alpha_1 in [True, False]:
        for alpha_2 in [True, False]:
            for alpha_3 in [True, False]:
                for alpha_4 in [True, False]:
                    truth_table.append([alpha_1, alpha_2, alpha_3, alpha_4, count_sat_assignments(alpha_1, alpha_2, alpha_3, alpha_4)])

    for row in truth_table:
        print(row)

#test_constraints(1, 2, 3)

In [23]:
print_debug = False

def test_3CNF_to_2CNF_conversion(clauses_all, num_vars):
    tree_cnf = CNF(from_clauses=clauses_all)
    two_cnf = CNF(from_clauses=convert_3CNF_to_2CNF(clauses_all, num_vars))
    two_cnf.nv = num_vars * num_vars * 4  # Update the number of variables in the 2CNF formula
        
    start = time.perf_counter()
    with Kissat404(bootstrap_with=tree_cnf) as solver:
        sat1 = solver.solve()
        if print_debug:
            print(f"3CNF: {sat1}. Solved in {1000 * (time.perf_counter() - start):.2f}ms")

    start = time.perf_counter()

    with Kissat404(bootstrap_with=two_cnf) as solver:
        sat2 = solver.solve()
        if print_debug:
            print(f"2CNF: {sat2}. Solved in {1000 * (time.perf_counter() - start):.2f}ms")

    if sat1 != sat2:
        print(f"==== Error: 3CNF and 2CNF results do not match for {entry.name}: 3CNF: {sat1}, 2CNF: {sat2}")
        print(f"3CNF clauses: {tree_cnf.clauses}")
        print(f"2CNF clauses: {two_cnf.clauses}")


def test_3CNF_to_2CNF_unsat_conversion(clauses_all, num_vars):
    for variable_1 in range(1, num_vars + 1):
        for variable_2 in range(variable_1 + 1, num_vars + 1):
            alpha_1 = get_combined_variable(variable_1, variable_2, num_vars)
            alpha_2 = get_combined_variable(-variable_1, variable_2, num_vars)
            alpha_3 = get_combined_variable(variable_1, -variable_2, num_vars)
            alpha_4 = get_combined_variable(-variable_1, -variable_2, num_vars)

            possible_forced = [alpha_1, alpha_2, alpha_3, alpha_4]
            count_sat = 0
            for forced_variable in possible_forced:
                two_cnf = CNF(from_clauses=convert_3CNF_to_2CNF(clauses_all, num_vars))
                two_cnf.nv = num_vars * num_vars * 4  # Update the number of variables in the 2CNF formula
                two_cnf.append([-forced_variable])  # Force the variable to be false


                with Kissat404(bootstrap_with=two_cnf) as solver:
                    sat = solver.solve()

                if sat:
                    count_sat += 1

                if count_sat == 0:
                    print(" +++ Found unsatisfiability")
                    return
                else:
                    print(f"Sat number: {count_sat}")

    print(" --- No unsatisfiability found for any forced variable combination")

max_tests = 1
for entry in os.scandir("../instances/random_unsat"):
    max_tests -= 1
    if (max_tests < 0):
        break
    if print_debug:
        print(f"Next: {entry.name}")
    num_vars, num_clauses, clauses = file_reader.read_cnf(entry.path)
    clauses_all = [clause.tolist() for clause in clauses]

    test_3CNF_to_2CNF_unsat_conversion(clauses_all, num_vars)


Sat number: 1
Sat number: 2
Sat number: 3
Sat number: 4
Sat number: 1
Sat number: 2
Sat number: 3
Sat number: 4
Sat number: 1
Sat number: 2
Sat number: 3
Sat number: 4
Sat number: 1
Sat number: 2
Sat number: 3
Sat number: 4
Sat number: 1
Sat number: 2
Sat number: 3
Sat number: 4
Sat number: 1
Sat number: 2
Sat number: 3
Sat number: 4
Sat number: 1
Sat number: 2
Sat number: 3
Sat number: 4
Sat number: 1
Sat number: 2
Sat number: 3
Sat number: 4
Sat number: 1
Sat number: 2
Sat number: 3
Sat number: 4
Sat number: 1
Sat number: 2
Sat number: 3
Sat number: 4
Sat number: 1
Sat number: 2
Sat number: 3
Sat number: 4
Sat number: 1
Sat number: 2
Sat number: 3
Sat number: 4
Sat number: 1
Sat number: 2
Sat number: 3
Sat number: 4
Sat number: 1
Sat number: 2
Sat number: 3
Sat number: 4
Sat number: 1
Sat number: 2
Sat number: 3
Sat number: 4
Sat number: 1
Sat number: 2
Sat number: 3
Sat number: 4
Sat number: 1
Sat number: 2
Sat number: 3
Sat number: 4
Sat number: 1
Sat number: 2
Sat number: 3
Sat nu